In [1]:
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

vectorstore = FAISS.from_texts(
    ["harrison worked at kensho"],
    embedding=OpenAIEmbeddings(),
)
retriever = vectorstore.as_retriever()
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

# The prompt expects input with keys for "context" and "question"
prompt = ChatPromptTemplate.from_template(template)

model = ChatOpenAI()

retrieval_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough(),
    }
    | prompt
    | model
    | StrOutputParser()
)

retrieval_chain.invoke("where did harrison work?")

'Harrison worked at Kensho.'

In [2]:
from operator import itemgetter

from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

vectorstore = FAISS.from_texts(
    ["harrison worked at kensho"],
    embedding=OpenAIEmbeddings(),
)
retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context:
{context}

Question: {question}

Answer in the following language: {language}
"""
prompt = ChatPromptTemplate.from_template(template)

chain = (
    {
        "context": itemgetter("question") | retriever,
        "question": itemgetter("question"),
        "language": itemgetter("language"),
    }
    | prompt
    | model
    | StrOutputParser()
)

chain.invoke(
    {
        "question": "where did harrison work",
        "language": "italian",
    }
)

'Harrison ha lavorato a Kensho.'

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel
from langchain_openai import ChatOpenAI

model = ChatOpenAI()
joke_chain = (
    ChatPromptTemplate.from_template(
        "tell me a joke about {topic}",
    )
    | model
)
poem_chain = (
    ChatPromptTemplate.from_template(
        "write a 2-line poem about {topic}",
    )
    | model
)

map_chain = RunnableParallel(
    joke=joke_chain,
    poem=poem_chain,
)

map_chain.invoke(
    {
        "topic": "bear",
    }
)

{'joke': AIMessage(content="Why don't bears wear shoes?\n\nBecause they already have bear feet!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 13, 'total_tokens': 27, 'completion_tokens_details': {'reasoning_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-069f61ea-e5d2-4cdf-afce-0b5402a6c3df-0', usage_metadata={'input_tokens': 13, 'output_tokens': 14, 'total_tokens': 27}),
 'poem': AIMessage(content='In the quiet forest, the bear roams free,\nA majestic creature, fierce and wild, yet gentle as can be.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 15, 'total_tokens': 40, 'completion_tokens_details': {'reasoning_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-9bfc935b-3b2f-4b7d-b1d6-dab1b05

In [4]:
%%timeit

joke_chain.invoke(
    {
        "topic": "bear",
    }
)

693 ms ± 74.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [5]:
%%timeit

poem_chain.invoke(
    {
        "topic": "bear",
    }
)

898 ms ± 359 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [6]:
%%timeit

map_chain.invoke(
    {
        "topic": "bear",
    }
)

801 ms ± 68.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
